# RAG Evaluation with DeepEval

This notebook evaluates the performance of Baseline RAG vs. RAG-Fusion using retrieval metrics (Recall@K, Precision@K, MRR@K) and generation metrics from **DeepEval** (Faithfulness, Answer Relevancy, Contextual Precision, Contextual Recall).

In [ ]:
# !pip install deepeval

import pandas as pd
import json
import os
import numpy as np
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualPrecisionMetric, ContextualRecallMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

In [ ]:
def load_jsonl(path):
    """Loads a JSONL file into a pandas DataFrame."""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return pd.DataFrame(data)

def extract_final_answer(full_text):
    """Trims LLM prompt and only takes the final answer."""
    if not isinstance(full_text, str):
        return ""
    
    # Token penanda di mana asisten mulai menjawab
    split_token = "<|im_start|>assistant\n"
    
    if split_token in full_text:
        # Ambil semua teks setelah token asisten
        return full_text.split(split_token)[-1].strip()
    return full_text.strip()

def calc_retrieval_metrics(row, retrieved_col, k=3):
    """Calculates Precision@K, Recall@K, and MRR@K."""
    retrieved_ids = row[retrieved_col][:k] # Potong hasil sesuai K
    ground_truth_ids = row['ground_truth_ids']
    
    gt_set = set(ground_truth_ids)
    
    # Deteksi mana yang "Hit" (Benar) di urutan top K
    hits = [1 if doc_id in gt_set else 0 for doc_id in retrieved_ids]
    
    # 1. Recall@K = (Jumlah Benar) / (Total Kunci Jawaban)
    recall = sum(hits) / len(gt_set) if len(gt_set) > 0 else 0.0
    
    # 2. Precision@K = (Jumlah Benar) / K
    precision = sum(hits) / k if k > 0 else 0.0
    
    # 3. MRR@K = 1 / (Ranking pertama yang benar)
    mrr = 0.0
    for i, hit in enumerate(hits):
        if hit == 1:
            mrr = 1.0 / (i + 1)
            break
            
    return pd.Series([precision, recall, mrr])

In [ ]:
K_VALUE = 3
FILEPATH = f"../knowledge-base/20260429_1829_rag_responses_dataset_k3.jsonl"

if not os.path.exists(FILEPATH):
    print(f"Error: File {FILEPATH} not found.")
else:
    print(f"1. Loading evaluation data from {FILEPATH}...")
    df = load_jsonl(FILEPATH)
    print(f"Total rows loaded: {len(df)}\n")

In [ ]:
# Handle nested baseline/fusion columns
if 'baseline' in df.columns and isinstance(df['baseline'].iloc[0], dict):
    df['baseline_answer'] = df['baseline'].apply(lambda x: x.get('answer') if isinstance(x, dict) else None)
    df['baseline_contexts_metadata'] = df['baseline'].apply(lambda x: x.get('metadata') if isinstance(x, dict) else None)
    df['baseline_retrieved_contexts'] = df['baseline'].apply(lambda x: x.get('contexts') if isinstance(x, dict) else None)
    
if 'fusion' in df.columns and isinstance(df['fusion'].iloc[0], dict):
    df['fusion_answer'] = df['fusion'].apply(lambda x: x.get('answer') if isinstance(x, dict) else None)
    df['fusion_contexts_metadata'] = df['fusion'].apply(lambda x: x.get('metadata') if isinstance(x, dict) else None)
    df['fusion_retrieved_contexts'] = df['fusion'].apply(lambda x: x.get('contexts') if isinstance(x, dict) else None)

# Extract Ground Truth IDs from metadata
df['ground_truth_ids'] = df['chunk_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)

# Extract Retrieval IDs from metadata
df['baseline_retrieved_ids'] = df['baseline_contexts_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)
df['fusion_retrieved_ids'] = df['fusion_contexts_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)

# Clean answers
print("Cleaning answers from prompts...")
df['clean_baseline_answer'] = df['baseline_answer'].apply(extract_final_answer)
df['clean_fusion_answer'] = df['fusion_answer'].apply(extract_final_answer)

In [ ]:
print(f"2. Calculating Retrieval Metrics for K={K_VALUE}...")

df[['baseline_Precision@K', 'baseline_Recall@K', 'baseline_MRR@K']] = df.apply(
    lambda row: calc_retrieval_metrics(row, 'baseline_retrieved_ids', k=K_VALUE), axis=1
)
df[['fusion_Precision@K', 'fusion_Recall@K', 'fusion_MRR@K']] = df.apply(
    lambda row: calc_retrieval_metrics(row, 'fusion_retrieved_ids', k=K_VALUE), axis=1
)

print("Retrieval Metrics Calculated.")

In [ ]:
print("\n3. Preparing data for DeepEval evaluation (LLM-as-a-Judge)...")

def format_contexts(contexts):
    if isinstance(contexts, list):
        return [getattr(c, 'page_content', str(c)) for c in contexts]
    return []

def create_test_cases(dataframe, answer_col, contexts_col):
    test_cases = []
    for i, row in dataframe.iterrows():
        test_case = LLMTestCase(
            input=row["question"],
            actual_output=row[answer_col],
            retrieval_context=format_contexts(row[contexts_col]),
            ground_truth=row["ground_truth"]
        )
        test_cases.append(test_case)
    return test_cases

test_cases_baseline = create_test_cases(df, "clean_baseline_answer", "baseline_retrieved_contexts")
test_cases_fusion = create_test_cases(df, "clean_fusion_answer", "fusion_retrieved_contexts")

model_name = "gpt-4o-mini"
metrics = [
    FaithfulnessMetric(threshold=0.5, model=model_name),
    AnswerRelevancyMetric(threshold=0.5, model=model_name),
    ContextualPrecisionMetric(threshold=0.5, model=model_name),
    ContextualRecallMetric(threshold=0.5, model=model_name)
]

print("\n> Running DeepEval Evaluation for BASELINE...")
evaluate(test_cases_baseline, metrics)
baseline_results = [ {m.__class__.__name__: m.score for m in tc.metrics} for tc in test_cases_baseline ]

print("\n> Running DeepEval Evaluation for FUSION...")
evaluate(test_cases_fusion, metrics)
fusion_results = [ {m.__class__.__name__: m.score for m in tc.metrics} for tc in test_cases_fusion ]

In [ ]:
print("\n" + "="*50)
print(f"FINAL EVALUATION RESULTS (Top-K = {K_VALUE})")
print("="*50)

res_b = pd.DataFrame(baseline_results)
res_f = pd.DataFrame(fusion_results)

summary_df = pd.DataFrame({
    "Metrik": [
        f"Recall (Manual)@{K_VALUE}", 
        f"Precision (Manual)@{K_VALUE}", 
        f"MRR (Manual)@{K_VALUE}", 
        "Faithfulness (DeepEval)", 
        "Answer Relevancy (DeepEval)",
        "Contextual Precision (DeepEval)",
        "Contextual Recall (DeepEval)"
    ],
    "RAG Baseline": [
        df['baseline_Recall@K'].mean(),
        df['baseline_Precision@K'].mean(),
        df['baseline_MRR@K'].mean(),
        res_b['FaithfulnessMetric'].mean(),
        res_b['AnswerRelevancyMetric'].mean(),
        res_b['ContextualPrecisionMetric'].mean(),
        res_b['ContextualRecallMetric'].mean()
    ],
    "RAG Fusion": [
        df['fusion_Recall@K'].mean(),
        df['fusion_Precision@K'].mean(),
        df['fusion_MRR@K'].mean(),
        res_f['FaithfulnessMetric'].mean(),
        res_f['AnswerRelevancyMetric'].mean(),
        res_f['ContextualPrecisionMetric'].mean(),
        res_f['ContextualRecallMetric'].mean()
    ]
})

summary_df